In [ ]:
import os
os.chdir('/content')

print("[DeFooocus] Preparing ...")

theme = "dark" #@param ["dark", "light"]
preset = "deafult" #@param ["deafult", "realistic", "anime", "lcm", "sai", "turbo", "lighting", "hypersd", "playground_v2.5", "dpo", "spo", "sd1.5"]
advenced_args = "--share --attention-split --always-cpu --all-in-fp32 --disable-offload-from-vram" #@param {type: "string"}

if preset != "deafult":
  args = f"{advenced_args} --theme {theme} --preset {preset}"
elif preset == "deafult":
  args = f"{advenced_args} --theme {theme}"

# Clean up previous clone completely
!rm -rf /content/DeFooocus
!git clone https://github.com/ehristoforu/DeFooocus.git
%cd /content/DeFooocus

# Upgrade pip first to ensure better wheel matching on Python 3.13
!pip install --upgrade pip

# Safely strip all references of pygit2 to fully bypass pip source compiling
!sed -i '/pygit2/d' requirements_versions.txt

# Modify requirements_versions.txt to remove strict version constraints that trigger source compilation
!sed -i 's/onnxruntime==1.18.1/onnxruntime/g' requirements_versions.txt
!sed -i 's/rembg==2.0.57/rembg/g' requirements_versions.txt
!sed -i '/safetensors/d' requirements_versions.txt
!sed -i '/tokenizers/d' requirements_versions.txt
!sed -i '/transformers/d' requirements_versions.txt
!sed -i '/numpy/d' requirements_versions.txt

# Explicitly install modern compatible versions first via binary wheels
!pip install --no-cache-dir --prefer-binary --upgrade torchsde tokenizers transformers

# Uninstall einops to resolve version conflict before installing requirements
!pip uninstall -y einops

# Install remaining requirements preferring binary wheels
!pip install --prefer-binary --no-cache-dir --upgrade -r requirements_versions.txt

# Deconstruct the exact Gradio source check directly on disk inside python packages
import gradio
gradio_dir = os.path.dirname(gradio.__file__)
for root, dirs, files in os.walk(gradio_dir):
    for file in files:
        if file.endswith('.py'):
            filepath = os.path.join(root, file)
            try:
                with open(filepath, 'r', encoding='utf-8') as f:
                    content = f.read()
                changed = False
                # Force is_bundled to always return True
                if 'def is_bundled' in content:
                    content = content.replace('def is_bundled()', 'def is_bundled():\n    return True\n    _unused')
                    changed = True
                # Search for the exact share error message raised in routing, and disable it
                if "Did you install Gradio from source files" in content:
                    # Safely replace the raising block with a neutral return or pass
                    content = content.replace(
                        "raise ValueError(\n            \"Did you install Gradio from source files? Share mode only works when Gradio is installed through the pip package.\"\n        ) from err",
                        "pass"
                    )
                    content = content.replace(
                        "raise ValueError(\n                \"Did you install Gradio from source files? Share mode only works when Gradio is installed through the pip package.\"\n            ) from err",
                        "pass"
                    )
                    content = content.replace(
                        "raise ValueError(\n                    \"Did you install Gradio from source files? Share mode only works when Gradio is installed through the pip package.\"\n                ) from err",
                        "pass"
                    )
                    changed = True
                if changed:
                    with open(filepath, 'w', encoding='utf-8') as f:
                        f.write(content)
            except Exception:
                pass

# Apply a comprehensive hotfix directly to Starlette and Jinja2 template handlers to resolve dict argument errors
with open('modules/ui_gradio_extensions.py', 'r') as f:
    ui_content = f.read()

jinja_starlette_hotfix = """import starlette.templating
import jinja2.utils
import gradio

# 0. Hotfix Gradio's internal package/bundling check so share mode is allowed
import gradio.utils
gradio.utils.is_bundled = lambda *args, **kwargs: True

# Also patch the actual exception-raising function inside gradio.routes if applicable
try:
    import gradio.routes
    # If Gradio attempts to raise a ValueError for non-pip installations, stub it out
    gradio.routes.is_bundled = lambda *args, **kwargs: True
except Exception:
    pass

# 1. Hotfix Starlette's template resolution to ensure 'name' is always a string
original_get_template = starlette.templating.Jinja2Templates.get_template

def safe_get_template(self, name, *args, **kwargs):
    if isinstance(name, dict):
        # Extract default index if dict configuration is erroneously passed as name
        name = name.get('template', 'index.html')
    return original_get_template(self, str(name), *args, **kwargs)

starlette.templating.Jinja2Templates.get_template = safe_get_template

# 2. Hotfix Jinja2's LRUCache to completely ignore unhashable keys (like nested dicts) without raising exceptions
original_getitem = jinja2.utils.LRUCache.__getitem__
original_setitem = jinja2.utils.LRUCache.__setitem__
original_contains = jinja2.utils.LRUCache.__contains__
original_get = jinja2.utils.LRUCache.get

def safe_getitem(self, key):
    try:
        return original_getitem(self, key)
    except TypeError:
        return None

def safe_setitem(self, key, value):
    try:
        original_setitem(self, key, value)
    except TypeError:
        pass

def safe_contains(self, key):
    try:
        return original_contains(self, key)
    except TypeError:
        return False

def safe_get(self, key, default=None):
    try:
        return original_get(self, key, default)
    except TypeError:
        return default

jinja2.utils.LRUCache.__getitem__ = safe_getitem
jinja2.utils.LRUCache.__setitem__ = safe_setitem
jinja2.utils.LRUCache.__contains__ = safe_contains
jinja2.utils.LRUCache.get = safe_get

# 3. Patch Starlette's TemplateResponse to sanitize context dicts
original_template_response = starlette.templating.Jinja2Templates.TemplateResponse

def safe_template_response(self, name, context, *args, **kwargs):
    if isinstance(name, dict):
        name = name.get('template', 'index.html')
    sanitized_context = {}
    for k, v in context.items():
        if isinstance(v, dict):
            class HashableDict(dict):
                def __hash__(self):
                    return hash(frozenset((kk, str(vv)) for kk, vv in self.items()))
            sanitized_context[k] = HashableDict(v)
        else:
            sanitized_context[k] = v
    return original_template_response(self, str(name), sanitized_context, *args, **kwargs)

starlette.templating.Jinja2Templates.TemplateResponse = safe_template_response
"""

# Insert this global catch-all hotfix at the very top of ui_gradio_extensions.py
ui_content = jinja_starlette_hotfix + "\n" + ui_content

with open('modules/ui_gradio_extensions.py', 'w') as f:
    f.write(ui_content)

# Patch patch_clip.py to fix AttributeError: module 'transformers.modeling_utils' has no attribute 'no_init_weights'
with open('modules/patch_clip.py', 'r') as f:
    clip_content = f.read()

# Add the compatibility context manager fallback at the very beginning of the file
clip_patch = """import contextlib
import transformers.modeling_utils as modeling_utils
if not hasattr(modeling_utils, 'no_init_weights'):
    @contextlib.contextmanager
    def dummy_no_init_weights():
        yield
    modeling_utils.no_init_weights = dummy_no_init_weights
"""

# Inject the monkeypatch at the top of the file to make sure it executes first
clip_content = clip_patch + "\n" + clip_content

# Safely bypass the missing nested text_model attribute on newer transformers CLIPTextModel instances
clip_content = clip_content.replace(
    "self.transformer.text_model.embeddings.to(torch.float32)",
    "(self.transformer.text_model if hasattr(self.transformer, 'text_model') else self.transformer).embeddings.to(torch.float32)"
)

with open('modules/patch_clip.py', 'w') as f:
    f.write(clip_content)

print("[DeFooocus] Starting ...")
# Bypass the updater launcher to avoid triggering the internal legacy pip install loop
!python launch.py $args

[DeFooocus] Preparing ...
Cloning into 'DeFooocus'...
remote: Enumerating objects: 7032, done.
remote: Total 7032 (delta 0), reused 0 (delta 0), pack-reused 7032 (from 1)
Receiving objects: 100% (7032/7032), 47.35 MiB | 31.40 MiB/s, done.
Resolving deltas: 100% (3825/3825), done.
/content/DeFooocus
